# 01 · 任务定义与数据探查（真实数据集）

> 升级版后训练项目的第 1 步。

## 这个项目在做什么
在垂直领域（**中文法律问答**）对 decoder-only 大模型做 **SFT + DPO 对齐**，并建立评测体系。

流程：`01 看数据 → 02 造数据 → 03 SFT → 04 DPO → 05 评测`

## 数据集：DISC-Law-SFT（复旦大学开源，真实数据）
- 来源：[ShengbinYue/DISC-Law-SFT](https://huggingface.co/datasets/ShengbinYue/DISC-Law-SFT)，法律问答子集约 **9.3 万条**真实法律咨询问答。
- 字段：`id` / `input`（问题）/ `output`（答案，多为几百字的专业解答，常引用具体法条）。
- 本 notebook 会**自动下载一份样本**（默认 2000 条）到 `data/`，下不动时自动回退到内置 `seed_clauses.jsonl`。

> 任务定义：**输入** = 用户法律问题；**输出** = 准确、有依据的解答。贴你在幂律的法律 AI 经历。

---


In [15]:
import os, json, sys   # os=文件/路径操作, json=读写json数据, sys=系统相关
import torch           # PyTorch：深度学习框架(模型和张量都靠它)

# 修证书：python.org 版 Python 默认找不到 CA 证书，会导致 https 下载报 CERTIFICATE_VERIFY_FAILED。
# 用 certifi 自带的证书清单顶上，本 notebook 所有 https 请求就都能验证通过了。
import certifi
os.environ["SSL_CERT_FILE"] = certifi.where()

# 自动找到项目根目录（notebooks 文件夹的上一级）
ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
DATA = os.path.join(ROOT, "data")      # data 目录完整路径
OUT  = os.path.join(ROOT, "outputs")   # outputs 目录完整路径
os.makedirs(OUT, exist_ok=True)        # 新建 outputs 目录；exist_ok=True=已存在也不报错

# 选计算设备：苹果芯片用 mps，英伟达显卡用 cuda，都没有就用 cpu
DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("ROOT  :", ROOT)
print("DEVICE:", DEVICE)

ROOT  : /Users/yunye/Documents/工作/项目/llm_sft_dpo_legalqa
DEVICE: mps


In [16]:
# ---- 下载真实数据集样本（首次运行才下载，之后直接用本地文件）----
import urllib.request

RAW = os.path.join(DATA, "disc_law_qa.jsonl")   # 原始数据存这里
N_DOWNLOAD = 5000                                # 只取前 N 条做这个小项目（全量 9.3 万太大）

if not os.path.exists(RAW):                       # 文件不存在才下载
    # 两个地址：国内镜像优先，失败再试官方
    urls = [
        "https://hf-mirror.com/datasets/ShengbinYue/DISC-Law-SFT/resolve/main/DISC-Law-SFT-Pair-QA-released.jsonl",
        "https://huggingface.co/datasets/ShengbinYue/DISC-Law-SFT/resolve/main/DISC-Law-SFT-Pair-QA-released.jsonl",
    ]
    for u in urls:
        try:
            print("下载中:", u)
            lines = []
            # urlopen 打开网络连接；for line in r 逐行读（边读边存，读够 N 条就停）
            with urllib.request.urlopen(u, timeout=60) as r:
                for line in r:
                    lines.append(line.decode("utf-8"))   # 网络读到的是字节，decode 转成文字
                    if len(lines) >= N_DOWNLOAD:
                        break
            open(RAW, "w", encoding="utf-8").writelines(lines)   # 写入本地文件
            print("已保存", len(lines), "条到", RAW)
            break                                     # 成功就跳出，不再试下一个地址
        except Exception as e:                        # 任何报错都接住，换下一个地址
            print("失败:", e)
else:
    print("已存在本地数据:", RAW)

下载中: https://hf-mirror.com/datasets/ShengbinYue/DISC-Law-SFT/resolve/main/DISC-Law-SFT-Pair-QA-released.jsonl
已保存 5000 条到 /Users/yunye/Documents/工作/项目/llm_sft_dpo_legalqa/data/disc_law_qa.jsonl


In [17]:
# ---- 读取 + 数据清洗 + 去重 → 整理成统一格式 ----
# ★ 二选一：有真实数据就用真实数据，没有(下载失败)才用内置种子集。
#   所以 raw 里要么全是真实数据、要么全是种子数据，绝不会两份掺在一起。
source = RAW if os.path.exists(RAW) else os.path.join(DATA, "seed_clauses.jsonl")
print("数据来源:", os.path.basename(source))

# 逐行读 jsonl（列表推导式：每行 json 文本 → 字典，组成字典列表）
raw = [json.loads(l) for l in open(source, encoding="utf-8") if l.strip()]

pool = []          # 清洗后的干净数据放这
seen = set()       # 记录见过的问题，用来去重（set 判断"在不在里面"很快）
for r in raw:
    # ★ 下面不是"合并两份数据"，而是"兼容两种字段名"：
    #   两个来源字段名不同——真实数据用 input/output，种子集用 question/answer。
    #   A or B 的规则：A 有值就用 A，A 为空/None 才用 B。
    #   所以同一条数据只会命中其中一个键（真实数据命中 input，种子集命中 question），不会混。
    q = (r.get("input") or r.get("question") or "").strip()    # .get(键):有就返回值,没有返回None(比 r["键"] 安全)；末尾 or "" 兜底成空串
    a = (r.get("output") or r.get("answer") or "").strip()
    if not q or not a:                  # 问题或答案为空 → 丢弃
        continue                        # continue=跳过这条，进下一条
    if len(a) < 15 or len(a) > 1000:    # 数据清洗：答案太短(没信息)或太长(训练慢) → 丢弃
        continue
    if q in seen:                       # 这个问题已出现过 → 去重丢弃
        continue
    seen.add(q)                         # 记下这个问题
    pool.append({"question": q, "answer": a})   # 留下，统一成 {question, answer}

QA_POOL = os.path.join(DATA, "qa_pool.json")
json.dump(pool, open(QA_POOL, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("清洗后剩余:", len(pool), "条  (原始", len(raw), "条)")

数据来源: disc_law_qa.jsonl
清洗后剩余: 4987 条  (原始 5000 条)


In [18]:
# ---- 看数据画像 + 样例 ----
import statistics as st
ql = [len(x["question"]) for x in pool]    # 每条问题字数
al = [len(x["answer"]) for x in pool]      # 每条答案字数
print("问题长度  平均%.0f  最长%d" % (st.mean(ql), max(ql)))
print("答案长度  平均%.0f  最短%d  最长%d" % (st.mean(al), min(al), max(al)))
print("=" * 60)
for x in pool[:2]:                          # 看前2条真实数据
    print("【问题】", x["question"])
    print("【答案】", x["answer"][:120], "...")   # 答案很长，只看前120字
    print("-" * 60)

问题长度  平均16  最长323
答案长度  平均457  最短15  最长999
【问题】 违章停车与违法停车是否有区别？
【答案】 对违反道路交通安全法律、法规关于机动车停放、临时停车规定的，可以指出违法行为，并予以口头警告，令其立即驶离。机动车驾驶人不在现场或者虽在现场但拒绝立即驶离，妨碍其他车辆、行人通行的处二十元以上二百元以下罚款。现在人们大多是称作违法停车，因此 ...
------------------------------------------------------------
【问题】 人民法院执行实际施工人对发包人的到期债权的，转包人或者违法分包人是否有权提异议？
【答案】 根据《最高人民法院新建设工程施工合同司法解释（一）理解与适用》，人民法院出版社出版在转包和违法分包的情况下，存在三方当事人，两个法律关系。一是承包人与发包人之间的建设工程施工合同关系；二是承包人作为转包人或者违法分包人与转包或者违法分包中的 ...
------------------------------------------------------------
